### Завдання 1: Завантаження VHI-індексів по областях
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. 
* При зберіганні файлу, до його імені потрібно додати дату та час завантаження. 
* Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [1]:
import urllib.request
import os
from datetime import datetime

data_dir = "vhi_data"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

def download_vhi_data(province_id, year1=1981, year2=2024):
    """Функція для завантаження VHI даних по одній області."""
    
    existing_files = [f for f in os.listdir(data_dir) if f.startswith(f"vhi_id_{province_id}_")]
    
    if existing_files:
        print(f"Область {province_id}: Дані вже існують ({existing_files[0]}). Пропускаємо завантаження.")
        return os.path.join(data_dir, existing_files[0])

    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1={year1}&year2={year2}&type=Mean"
    
    now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"vhi_id_{province_id}_{now}.csv"
    filepath = os.path.join(data_dir, filename)
    
    try:
        with urllib.request.urlopen(url) as response:
            html = response.read()
            
        with open(filepath, 'wb') as f:
            f.write(html)
            
        print(f"Область {province_id}: Успішно завантажено у файл {filename}")
        return filepath
        
    except Exception as e:
        print(f"Область {province_id}: Помилка завантаження! ({e})")
        return None

print("Починаємо перевірку та завантаження даних...\n")
for i in range(1, 28):
    download_vhi_data(i)
print("\nПроцес завершено!")

Починаємо перевірку та завантаження даних...

Область 1: Успішно завантажено у файл vhi_id_1_2026-03-06_20-02-59.csv
Область 2: Успішно завантажено у файл vhi_id_2_2026-03-06_20-03-03.csv
Область 3: Успішно завантажено у файл vhi_id_3_2026-03-06_20-03-04.csv
Область 4: Успішно завантажено у файл vhi_id_4_2026-03-06_20-03-05.csv
Область 5: Успішно завантажено у файл vhi_id_5_2026-03-06_20-03-06.csv
Область 6: Успішно завантажено у файл vhi_id_6_2026-03-06_20-03-07.csv
Область 7: Успішно завантажено у файл vhi_id_7_2026-03-06_20-03-08.csv
Область 8: Успішно завантажено у файл vhi_id_8_2026-03-06_20-03-10.csv
Область 9: Успішно завантажено у файл vhi_id_9_2026-03-06_20-03-11.csv
Область 10: Успішно завантажено у файл vhi_id_10_2026-03-06_20-03-12.csv
Область 11: Успішно завантажено у файл vhi_id_11_2026-03-06_20-03-13.csv
Область 12: Успішно завантажено у файл vhi_id_12_2026-03-06_20-03-14.csv
Область 13: Успішно завантажено у файл vhi_id_13_2026-03-06_20-03-16.csv
Область 14: Успішно зав

### Завдання 2: Зчитування та очистка даних (Data Cleaning), зміна індексів
* Зчитати завантажені текстові файли у pandas dataframe. 
* Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. 
* Додати стовпчики з назвою та індексом області.
* Реалізувати процедуру зміни індексів (щоб 1 область була Вінницька).

In [2]:
import pandas as pd
import os

# Формат словника: старий_id: (новий_id, 'Назва області')
noaa_to_ukr = {
    1: (22, 'Черкаська'), 2: (24, 'Чернігівська'), 3: (23, 'Чернівецька'), 
    4: (25, 'АР Крим'), 5: (3, 'Дніпропетровська'), 6: (4, 'Донецька'), 
    7: (8, 'Івано-Франківська'), 8: (19, 'Харківська'), 9: (20, 'Херсонська'), 
    10: (21, 'Хмельницька'), 11: (9, 'Київська'), 12: (26, 'м. Київ'), 
    13: (10, 'Кіровоградська'), 14: (11, 'Луганська'), 15: (12, 'Львівська'), 
    16: (13, 'Миколаївська'), 17: (14, 'Одеська'), 18: (15, 'Полтавська'), 
    19: (16, 'Рівненська'), 20: (27, 'м. Севастополь'), 21: (17, 'Сумська'), 
    22: (18, 'Тернопільська'), 23: (6, 'Закарпатська'), 24: (1, 'Вінницька'), 
    25: (2, 'Волинська'), 26: (7, 'Запорізька'), 27: (5, 'Житомирська')
}

def clean_and_combine_data(data_dir="vhi_data"):
    all_dfs = []
    
    for filename in os.listdir(data_dir):
        if filename.endswith(".csv"):
            filepath = os.path.join(data_dir, filename)
            
            old_id = int(filename.split('_')[2])
            new_id, prov_name = noaa_to_ukr[old_id]
            
            df = pd.read_csv(filepath, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty'])
            
            df = df.drop('empty', axis=1)
            
            df = df.dropna()
            
            df['Year'] = df['Year'].astype(str).str.replace(r'<.*?>', '', regex=True)
            df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
            df = df.dropna(subset=['Year'])

            df = df.astype({'Year': int, 'Week': int, 'SMN': float, 'SMT': float, 'VCI': float, 'TCI': float, 'VHI': float})
            
            df = df[df['VHI'] != -1]
            
            df['Province_ID'] = new_id
            df['Province_Name'] = prov_name
            
            all_dfs.append(df)
            
    full_df = pd.concat(all_dfs, ignore_index=True)
    return full_df

df = clean_and_combine_data()

print("Дані успішно очищено та об'єднано!")
print(f"Загальна кількість записів: {len(df)}")
df.head()

Дані успішно очищено та об'єднано!
Загальна кількість записів: 59022


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,1,0.053,260.31,45.01,39.46,42.23,22,Черкаська
1,1982,2,0.054,262.29,46.83,31.75,39.29,22,Черкаська
2,1982,3,0.055,263.82,48.13,27.24,37.68,22,Черкаська
3,1982,4,0.053,265.33,46.09,23.91,35.00,22,Черкаська
4,1982,5,0.050,265.66,41.46,26.65,34.06,22,Черкаська


### Завдання 3.1: Ряд VHI для області за вказаний рік
Реалізувати процедуру для формування вибірки ряду VHI для заданої області та вказаного року. Функція має виводити читабельний результат.

In [3]:
def get_vhi_by_province_and_year(df, province_id, year):
    """Повертає та друкує ряд VHI для вказаної області та року."""
    
    filtered_df = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    
    if filtered_df.empty:
        print(f"Даних для області {province_id} за {year} рік не знайдено.")
        return None
    
    prov_name = filtered_df['Province_Name'].iloc[0]
    
    print(f"--- Дані VHI: {prov_name} область (ID: {province_id}), {year} рік ---")
    
    result = filtered_df[['Week', 'VHI']].reset_index(drop=True)
    return result

get_vhi_by_province_and_year(df, 1, 2015)

--- Дані VHI: Вінницька область (ID: 1), 2015 рік ---


,Week,VHI
0,1,47.22
1,2,49.81
2,3,50.72
3,4,48.89
4,5,46.44
5,6,44.18
6,7,42.01
7,8,41.02
8,9,42.36
9,10,43.78


### Завдання 3.2: Ряд VHI за вказаний діапазон років для вказаних областей
Реалізувати процедуру для формування вибірки ряду VHI за вказаний діапазон років для заданого списку областей.

In [4]:
def get_vhi_by_provinces_and_years(df, province_ids, start_year, end_year):
    """Повертає вибірку VHI за діапазон років для заданого списку областей."""
    
    filtered_df = df[
        (df['Province_ID'].isin(province_ids)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ]
    
    if filtered_df.empty:
        print("За вказаними параметрами даних не знайдено.")
        return None
    
    found_provinces = ", ".join(filtered_df['Province_Name'].unique())
    print(f"--- Дані VHI для: {found_provinces} ---")
    print(f"--- Період: {start_year}-{end_year} рр. ---")
    
    result = filtered_df[['Year', 'Week', 'Province_Name', 'VHI']].reset_index(drop=True)
    return result

get_vhi_by_provinces_and_years(df, [1, 9], 2010, 2012)

--- Дані VHI для: Київська, Вінницька ---
--- Період: 2010-2012 рр. ---


,Year,Week,Province_Name,VHI
0,2010,1,Київська,47.84
1,2010,2,Київська,47.23
2,2010,3,Київська,49.70
3,2010,4,Київська,52.06
4,2010,5,Київська,52.79
...,...,...,...,...
307,2012,48,Вінницька,55.52
308,2012,49,Вінницька,60.67
309,2012,50,Вінницька,61.06
310,2012,51,Вінницька,59.49


### Завдання 3.3: Статистика VHI (Екстремуми, середнє, медіана)
Реалізувати процедуру для пошуку екстремумів (min та max), середнього та медіани для вказаних областей та років.

In [5]:
def get_vhi_stats(df, province_ids, start_year, end_year):
    """Повертає статистику (min, max, mean, median) VHI для заданих областей та років."""
    
    filtered_df = df[
        (df['Province_ID'].isin(province_ids)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ]
    
    if filtered_df.empty:
        print("За вказаними параметрами даних не знайдено.")
        return None
    
    print("--- Статистика VHI (Мінімум, Максимум, Середнє, Медіана) ---")
    
    stats = filtered_df.groupby(['Province_Name', 'Year'])['VHI'].agg(['min', 'max', 'mean', 'median']).reset_index()
    
    stats.columns = ['Область', 'Рік', 'Min_VHI', 'Max_VHI', 'Середнє_VHI', 'Медіана_VHI']
    
    stats = stats.round(2)
    
    return stats

get_vhi_stats(df, [1, 9], 2010, 2015)

--- Статистика VHI (Мінімум, Максимум, Середнє, Медіана) ---


,Область,Рік,Min_VHI,Max_VHI,Середнє_VHI,Медіана_VHI
0,Вінницька,2010,38.12,62.71,50.10,51.34
1,Вінницька,2011,36.40,68.10,46.80,42.48
2,Вінницька,2012,29.28,61.06,40.83,39.41
3,Вінницька,2013,32.92,63.58,52.02,51.54
4,Вінницька,2014,36.37,66.94,51.38,48.20
5,Вінницька,2015,19.94,50.72,38.94,41.92
6,Київська,2010,26.82,52.79,40.70,40.70
7,Київська,2011,31.72,74.38,47.12,44.07
8,Київська,2012,38.10,54.13,46.33,45.52
9,Київська,2013,30.26,59.15,48.29,47.61
